# 03 — Model Training
Train models for different markets using XGBoost / LightGBM

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, brier_score_loss

from config import PATHS, MODEL_PARAMS, RANDOM_SEED
print("✅ Imports done")

In [ ]:
# Load feature set
df = pd.read_parquet(PATHS.PROCESSED / "features_master.parquet")
print(f"Loaded {len(df):,} matches")

# Use only recent seasons for faster training first
df = df[df['season'].isin(['2324', '2425'])].copy()

In [ ]:
# === MARKET 1: Match Result (1X2) ===
feature_cols = [col for col in df.columns if col not in 
               ['result', 'date', 'home_team', 'away_team', 'league_key', 'season', 'season_label']]

X = df[feature_cols].fillna(-999)
y = df['result'].map({'H': 0, 'D': 1, 'A': 2})

print("Training Match Result model...")

tscv = TimeSeriesSplit(n_splits=5)
models = []

for train_idx, val_idx in tscv.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(**MODEL_PARAMS.XGBOOST_DEFAULT)
    model.fit(X_train, y_train)
    models.append(model)
    
    pred = model.predict(X_val)
    print(f"Fold accuracy: {accuracy_score(y_val, pred):.4f}")

# Save model
joblib.dump(models, PATHS.MODEL_MATCH_RESULT)
print("✅ Match Result model saved!")